# PIF Normalisation

## Stage 1: Intra-sensor PIF normalisation

Run MDPS-cylinder PIF normalisation **separately within each sensor group** (WV2 and SD8).
This removes inter-date atmospheric/illumination variation within each sensor without mixing
spectral responses between sensors.

As opposed to MAD and IR-MAD, which are pairwise, this is specifically a multidate algorithm
that attempts to find PIFs common to all scenes via the MPDS cylinder. Here we use land pixels
as PIFs, because the water pixels are "contaminated" by both water column attenuation and
atmospheric composition with no way to distinguish between the two.

How the algorithm works:

**1. Random sample**
Random sample from the training dataset (valid, non-saturated, non-edge land pixels).

**2. Pairwise spectral angle mapping**
Selects the image with minimum mean angular distance (the medoid) as the reference.

**3. Build an MDPS cylinder per-band**
   1. PCA across image axes -> principal axis of interdate variation
   2. Project each pixel onto axis; pixels close to axis vary consistently across dates
   3. Grow cylinder until user-defined fraction of valid sample pixels fall inside (PIFs for that band)

**4. RANSAC per band**
Fits linear gain/offset on PIF pixels; inliers are the final normalisation PIFs.

**5. Apply gain/offset to the full image**

Citations: Schott et al. (1988); Paolini et al. (2006); Du et al. (2002); Bao et al. (2012); Xu et al.

<!-- ## Stage 2: Cross-sensor spectral harmonisation

After intra-sensor PIF, SD8 bands are still in SD8 spectral response space. Apply the empirical coefficients from 05_0 to convert SD8 to WV2 SR. -->

In [3]:
from pathlib import Path
import pystac

import pandas as pd
import pickle
import xarray as xr
import rioxarray as rxr
import numpy as np
import rasterio
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from dataclasses import dataclass
from functools import partial

from rasterio.enums import Resampling

from tqdm.notebook import tqdm

from sklearn.linear_model import RANSACRegressor, LinearRegression

In [4]:
import sys
sys.path.insert(0, "..")

In [5]:
from config import *

# -- Catalog -----------------------------------------------------------------------------------
CATALOG             = load_catalog()
INPUT_ASSET         = {
                        "wv2-imagery" : "harm",
                        "sd8-imagery" : "harm"
                    }

# -- Bands -------------------------------------------------------------------------------------
NORM_BANDS = {
    "wv2-imagery": ([1, 2, 3, 4], ["B", "G", "Y", "R"]),
    "sd8-imagery": ([1, 2, 3, 4], ["B", "G", "Y", "R"]),
}

PASSTHROUGH = {
    "wv2-imagery": ([], []),
    "sd8-imagery": ([], [])
}

# -- Normalisation Parameters ----------------------------------------------------------------------
FIT_SAMPLE          = 1.0               # Pixels to use for fitting, can be percentage or int
CYLINDER_MAX_GROWTH = 1.5               # Max cylinder size % above initial
CYLINDER_GROW_RATE  = 0.01              # N% step to increase until pif_fraction is reached
PIF_FRACTION        = 0.01              # Best N% of training samples to use as PIFS 

# -- Output Options ----------------------------------------------------------------------------
OUT_DIR             = Path("../out/norm")
MKDIR               = True
OUT_NAME            = "norm"
OUT_EXT             = "tif"

# -- Execution Flags ---------------------------------------------------------------------------
OVERWRITE           = True

In [7]:
wv2_col  = CATALOG.get_child("wv2-imagery")
sd8_col  = CATALOG.get_child("sd8-imagery")
wv2_items = list(wv2_col.get_items())
sd8_items = list(sd8_col.get_items())
all_items = wv2_items + sd8_items

for item in all_items:
    tqdm.write(f"ID: {item.id} | Date: {item.datetime} | Path: {item.assets[INPUT_ASSET[item.collection_id]].href}")

ID: wv2-20110610 | Date: 2011-06-10 00:30:00+00:00 | Path: C:/Users/samla/study/25-SEM3.ENVM7133/ENVM7133/out/harm/wv2-20110610-harm.tif
ID: wv2-20120620 | Date: 2012-06-20 00:30:00+00:00 | Path: C:/Users/samla/study/25-SEM3.ENVM7133/ENVM7133/out/harm/wv2-20120620-harm.tif
ID: wv2-20140714 | Date: 2014-07-14 00:30:00+00:00 | Path: C:/Users/samla/study/25-SEM3.ENVM7133/ENVM7133/out/harm/wv2-20140714-harm.tif
ID: wv2-20150701 | Date: 2015-07-01 00:30:00+00:00 | Path: C:/Users/samla/study/25-SEM3.ENVM7133/ENVM7133/out/harm/wv2-20150701-harm.tif
ID: sd8-20220713 | Date: 2022-07-13 23:13:22.862989+00:00 | Path: C:/Users/samla/study/25-SEM3.ENVM7133/ENVM7133/out/harm/sd8-20220713-harm.tif
ID: sd8-20230706 | Date: 2023-07-06 23:22:24.866083+00:00 | Path: C:/Users/samla/study/25-SEM3.ENVM7133/ENVM7133/out/harm/sd8-20230706-harm.tif
ID: sd8-20240722 | Date: 2024-07-22 23:13:19.455922+00:00 | Path: C:/Users/samla/study/25-SEM3.ENVM7133/ENVM7133/out/harm/sd8-20240722-harm.tif
ID: sd8-20250610 | D

# 1. Intra-sensor Normalisation

Pipeline:
1. mdps cylinder (Paolini et al 2006)
   1. n-dimensional PCA -> project each pixel onto major axis -> keep pixels
   2. within cylinder radius u (iterate u until >= pif_fraction pixels)
2. Xu iterative regression refinement?
   1. per-band: fit ols -> remove points with d>0.2 x d-max -> repeat until 1) >95% of points lie within 95% CI and 2) d-max < 1.5 x d-mean
   2. intersect across all bands -> final pifs
3. RANSAC regression on final pifs

In [5]:
@dataclass
class BandResult:
    band:      str | int   # named coordinate value or integer index
    gain:      float
    offset:    float
    r2:        float
    n_pifs:    int

    def __repr__(self):
        return (
            f"Band {self.band!s:>6} | "
            f"gain = {self.gain:.4f} | "
            f"offset = {self.offset:+.4f} | "
            f"r2 = {self.r2:.4f} | "
            f"n PIFs = {self.n_pifs:,}"
        )

In [6]:
class PIFNormaliser:
    def __init__(
        self,
        trainers: list,
        targets: list | None = None,
        pif_fraction: float = 0.10,
        cylinder_max_growth = 2.0,
        cylinder_grow_rate = 0.01,
        fit_sample = 100_000,
        random_state = 42,
        ref_idx: int | None = None,
        verbose = False
    ):

        # Img number check
        if len(trainers) < 2:
            raise ValueError(f"Need at least 2 images. Got {len(trainers)}")

        # Img shape check
        shapes = [da.shape for da in trainers]
        if len(set(shapes)) != 1:
            raise ValueError(f"All images must have the same shape. Got {shapes}")

        # Variables
        self.trainers = trainers
        self.targets = targets if targets is not None else trainers
        self.pif_fraction = pif_fraction
        self.fit_sample = fit_sample
        self.random_state = random_state
        self.cylinder_max_growth = cylinder_max_growth
        self.cylinder_grow_rate = cylinder_grow_rate
        self.ref_idx = ref_idx
        self._n_images = len(trainers)
        self._n_bands = trainers[0].shape[0]
        self._band_labels = self._get_band_labels(trainers[0])

        # Results from normalise
        self.normalised: list | None = None
        self.results: dict | None = None
        self._pif_mask: np.ndarray | None = None
        self._samples: np.ndarray | None = None
        self._sample_coords: tuple | None = None
        self._mdps_cache: dict | None = None   # {b: (proj_1d, dist, U, valid)} for plot_mdps

        # Verbose
        self.verbose = verbose
        self._report_lines = []

    # ============================================================================
    # 1. HELPERS
    # ============================================================================
    
    @staticmethod
    def _get_band_labels(da):
        coords = da.coords.get("band")
        if coords is not None:
            return list(coords.values)
        return list(range(1, da.shape[0] + 1))

    def _log(self, msg: str):
        self._report_lines.append(msg)
        if self.verbose:
            tqdm.write(msg)

    # ============================================================================
    # 2. SAMPLING
    # ============================================================================
    
    def _sample(self):
        n_bands, ny, nx = self.trainers[0].shape
        n_pixels = ny * nx
        rng = np.random.default_rng(self.random_state)

        # --- Build candidate mask ---
        self._log("Building candidate mask from training data...")
        candidate_1d = np.isfinite(self.trainers[0].isel(band=0).values).ravel()
        candidate_idx = np.where(candidate_1d)[0]
        self.n_candidates = len(candidate_idx)
        self._log(f"Candidate pixels: {self.n_candidates:,} / {n_pixels,} ({100.*self.n_candidates / n_pixels:.1f}%)")

        # --- Either use integer or percentage for samplign ---
        if isinstance(self.fit_sample, float):
            self.n_sample = max(1, int(np.ceil(self.fit_sample * self.n_candidates)))
        else:
            self.n_sample = self.fit_sample
        flat_idx = rng.choice(candidate_idx, size=min(self.n_sample, self.n_candidates), replace=False)
        flat_idx.sort()
        iy, ix = np.unravel_index(flat_idx, (ny, nx))
        self._sample_coords = (iy, ix)
        
        # --- Pixel values ---
        raw = np.full((self._n_images, n_bands, len(flat_idx)), np.nan, dtype=np.float32)
        pbar_da = tqdm(self.trainers, desc = "Reading images", unit = "img", dynamic_ncols = True)
        for i, da in enumerate(pbar_da):
            pbar_da.set_description(f"Reading image {i + 1}")
            pbar_b = tqdm(range(n_bands), desc = f"Reading bands", unit = "band", dynamic_ncols = True, leave = False)
            for i_b, b in enumerate(pbar_b):
                pbar_b.set_description(f"Reading band {i_b + 1}")
                band_arr = da.isel(band = b).values
                raw[i, b, :] = band_arr[iy, ix]
        
        # --- Report valid pxs ---
        for b in range(n_bands):
            valid_b = np.all(np.isfinite(raw[:, b, :]), axis=0)
            self._log(f"Band {self._band_labels[b]!s:>6}: {valid_b.sum():,} / {len(flat_idx):,} valid pixels")

        return raw

    def _select_medoid(self, samples):
        N, B, P = samples.shape

        # --- Filter to jointly valid pixels for MPDS ---
        joint_valid = np.all(np.isfinite(samples.reshape(N * B, P)), axis=0)
        s = samples[:, :, joint_valid]
        norms = np.linalg.norm(s, axis = 1, keepdims = True) + 1e-10
        unit = s / norms

        # --- Perform spectral angle mapping pairwise across all images ---
        n_pairs = N * (N - 1) // 2
        pbar_sam = tqdm(total = n_pairs, desc = "Pairwise SAM", unit = "pair", dynamic_ncols = True)
        dist = np.zeros((N, N), dtype = np.float64)
        for i in range(N):
            for j in range(i + 1, N):
                cos_sim = np.clip((unit[i] * unit[j]).sum(axis=0), -1.0, 1.0)
                sam = np.arccos(cos_sim).mean()
                dist[i, j] = sam
                dist[j, i] = sam
                pbar_sam.update(1)
        pbar_sam.close()

        # --- Get medoid as img having lowest mean spectral angle ---
        ref_idx = int(dist.sum(axis=1).argmin())

        df = pd.DataFrame(
            dist,
            index = [f"img_{i}" for i in range(N)],
            columns = [f"img_{j}" for j in range(N)]
        ).round(4)

        df["mean_SAM"] = dist.sum(axis = 1) / (N - 1)
        df.index = [f"{idx} <-- ref" if i == ref_idx else idx for i, idx in enumerate(df.index)]

        self._log("\nMedoid Seference Selection")
        self._log(df.to_string())
        self._log(f"\nSelected image {ref_idx} (mean SAM = {dist.sum(axis=1)[ref_idx] / (N-1):.4f} rad)\n")

        self._medoid_df = df
        return ref_idx

    # ============================================================================
    # 3. MPDS
    # ============================================================================
    
    def _cylinder_for_band(self, band_samples):

        # --- Valid pixels ---
        valid = np.all(np.isfinite(band_samples), axis=0)
        X = band_samples[:, valid].T.astype(np.float64)
        n_pixels_valid = X.shape[0]

        # --- Standardise to Z ---
        mu = X.mean(axis = 0)
        std = X.std(axis = 0) + 1e-10
        X_z = (X - mu) / std

        # --- PCA ---
        pca = PCA(n_components = 1)
        pca.fit(X_z)
        axis_dir = pca.components_[0]

        # --- Paolini et al eq. 13 ---
        proj = X_z @ axis_dir
        S = np.outer(proj, axis_dir)
        dist = np.sqrt(((X_z - S) ** 2).sum(axis = 1))

        # --- Grow cylinder to select PIFs ---
        target_n = max(int(np.ceil(self.pif_fraction * n_pixels_valid)), 2)
        U = np.percentile(dist, self.pif_fraction * 100)
        U_max = U * self.cylinder_max_growth

        with tqdm(desc="Growing cylinder", leave=False, dynamic_ncols=True, unit="step") as pbar_cyl:
            for _ in range(500):
                inside = (dist <= U).sum()
                pbar_cyl.set_postfix(inside=f"{inside:,}", target=f"{target_n:,}", U=f"{U:.4f}")
                pbar_cyl.update(1)
                if inside >= target_n:
                    break
                if U >= U_max:
                    self._log(f"Cylinder growth capped at {self.cylinder_max_growth:.1f}x")
                    break
                U *= (1 + self.cylinder_grow_rate)

        # --- Map back to valid pixels ---
        result = np.zeros(band_samples.shape[1], dtype=bool)
        result[valid] = (dist <= U)
        return result, proj, dist, U, valid

    def _mdps_select(self, samples):
        masks = {}
        cache = {}

        pbar_b = tqdm(range(self._n_bands), desc = "MDPS for band", unit = "band")
        for b in pbar_b:

            # --- Run cylinder selection ---
            band_mask, proj_1d, dist, U, valid = self._cylinder_for_band(samples[:, b, :])
            masks[b] = band_mask
            cache[b] = (proj_1d, dist, U, valid)
            self._log(f"Band {self._band_labels[b]!s:<10} | {band_mask.sum():,} px ({100*band_mask.mean():.1f}%)")

        self._mdps_cache = cache
        return masks

    # ============================================================================
    # 4. RANSAC
    # ============================================================================
    @staticmethod
    def _ransac(ref_pifs, tgt_pifs, residual_threshold=None, n_iter=100, min_samples=2, random_state = 42):
        X = tgt_pifs.reshape(-1, 1).astype(np.float64)
        y = ref_pifs.astype(np.float64)

        # --- Fit RANSAC ---
        ransac = RANSACRegressor(
            estimator=LinearRegression(fit_intercept=True),
            min_samples=min_samples,
            residual_threshold=residual_threshold,
            max_trials=n_iter,
            random_state=random_state,
        )
        ransac.fit(X, y)

        # --- Extract coefficients ---
        gain   = float(ransac.estimator_.coef_[0])
        offset = float(ransac.estimator_.intercept_)

        # --- Report R2 and stats on inliers ---
        inlier_mask = ransac.inlier_mask_
        predicted = gain * tgt_pifs[inlier_mask] + offset
        ss_res = float(((ref_pifs[inlier_mask] - predicted) ** 2).sum())
        ss_tot = float(((ref_pifs[inlier_mask] - ref_pifs[inlier_mask].mean()) ** 2).sum()) + 1e-10
        r2  = float(1.0 - ss_res / ss_tot)
        n_inliers = int(inlier_mask.sum())

        return gain, offset, r2, n_inliers

    # ============================================================================
    # 5. TRANSFORM
    # ============================================================================
    @staticmethod
    def _transform(da, gains, offsets):
        g = xr.DataArray(gains.astype(np.float32), dims = ["band"])
        o = xr.DataArray(offsets.astype(np.float32), dims = ["band"])

        out = (da * g + o).clip(min = 0)
        out.attrs = da.attrs.copy()
        out.name  = (da.name or "image") + "_norm"

        if da.rio.crs is not None:
            out = out.rio.write_crs(da.rio.crs)
        return out

    @staticmethod
    def _apply_gains_np(sample, gains, offsets):
        return np.clip(
            gains[:, np.newaxis] * sample + offsets[:, np.newaxis],
            0, None
        ).astype(np.float32)

    # ============================================================================
    # 6. EXECUTE
    # ============================================================================
    def normalise(self):
        pbar = tqdm(total = 4, dynamic_ncols = True)

        # ----------------------------------------------------------------------------
        # 1. SAMPLE
        # ----------------------------------------------------------------------------
        sample_desc = f"{self.fit_sample*100:.0f}%" if isinstance(self.fit_sample, float) else f"{self.fit_sample:,}"
        pbar.set_description(f"[1/4] Sampling {sample_desc} of pixels across {self._n_images} images")
        samples = self._sample()
        self._samples = samples
        pbar.update(1)

        # ----------------------------------------------------------------------------
        # 2. Medoid
        # ----------------------------------------------------------------------------
        if self.ref_idx is None:
            pbar.set_description(f"[2/4] Getting medoid from {self._n_images} images")
            self.ref_idx = self._select_medoid(samples)
        else:
            pbar.set_description(f"[2/4] Using fixed ref_idx = {self.ref_idx}")
            self._log(f" Fixed reference: image {self.ref_idx}")
        pbar.update(1)

        # ----------------------------------------------------------------------------
        # 3. MDPS
        # ----------------------------------------------------------------------------
        pbar.set_description("[3/4] MDPS cylinder selection")
        self._log(f"\nMDPS selection  ({self._n_images} images, {self._n_bands} bands)")
        pif_masks = self._mdps_select(samples)
        self._pif_mask = pif_masks
        pbar.update(1)

        # ----------------------------------------------------------------------------
        # 4. RANSAC
        # ----------------------------------------------------------------------------
        pbar.set_description("[4/4] Fitting gain / offset per band")
        results    = {}
        normalised = [None] * self._n_images

        normalised[self.ref_idx] = self.targets[self.ref_idx]

        pbar_i = tqdm(enumerate(self.trainers), desc = f"Normalising images", unit = "img", total = self._n_images, dynamic_ncols = True)
        for i, da in pbar_i:
            pbar_i.set_description(f"Normalising images {i+1}")
            if i == self.ref_idx:
                continue

            gains   = np.zeros(self._n_bands, dtype = np.float64)
            offsets = np.zeros(self._n_bands, dtype = np.float64)
            band_results = []

            pbar_b = tqdm(range(self._n_bands), desc = f"Normalising bands", unit = "band", dynamic_ncols = True, leave = False)
            for b in pbar_b:
                pbar_b.set_description(f"Normalising bands {i+1}")
                mask     = pif_masks[b]
                ref_pifs = samples[self.ref_idx][b][mask]
                tgt_pifs = samples[i][b][mask]

                both_finite = np.isfinite(ref_pifs) & np.isfinite(tgt_pifs)
                ref_pifs = ref_pifs[both_finite]
                tgt_pifs = tgt_pifs[both_finite]

                g, o, r2, n_inliers = self._ransac(ref_pifs, tgt_pifs, random_state=self.random_state)
                gains[b]   = g
                offsets[b] = o
                band_results.append(BandResult(
                    band   = self._band_labels[b],
                    gain   = g,
                    offset = o,
                    r2     = r2,
                    n_pifs = n_inliers,
                ))
                self._log(f"Image {i+1} | Band {self._band_labels[b]!s:>6}, gain = {g:.4f}, offset = {o:+.4f}, r2 = {r2:.4f}")

            results[i]    = band_results
            normalised[i] = self._transform(self.targets[i], gains, offsets)

        self.results = results
        self.normalised = normalised
        pbar.update(1)
        pbar.close()

        return normalised

    # ============================================================================
    # 7. DIAG
    # ============================================================================
    def _check_fitted(self):
        if self.normalised is None:
            raise RuntimeError("Call .normalise() first.")

    def log(self):
        self._check_fitted()
        print("\n".join(self._report_lines))

    def summary(self):
        self._check_fitted()
        lines = [f"\nPIF Normalisation Summary | n = {self._n_images} | ref_idx = {self.ref_idx} | candidates = {self.n_candidates} | sampled = {self.n_sample}\n"]
        
        for b, mask in self._pif_mask.items():
            lines.append(f"{self._band_labels[b]!s:<10} | {mask.sum():,} PIFs ({100 * mask.mean():.1f} %)")
        lines.append("")

        for i, band_results in self.results.items():
            lines.append(f"Image {i+1}:")
            for r in band_results:
                lines.append(f"{r!r}")
        return "\n".join(lines)

    def plot_diag(self, output = None, sample = 5_000):
        self._check_fitted()

        rng    = np.random.default_rng(self.random_state)
        n_samp = min(sample, self._samples.shape[2])
        idx    = rng.choice(self._samples.shape[2], n_samp, replace = False)

        pbar_i = tqdm(self.results.items(), desc = "Plotting images", dynamic_ncols = True)
        for i, band_results in pbar_i:
            ncols = min(self._n_bands, 4)
            nrows = 2 * ((self._n_bands + ncols - 1) // ncols)
            fig, axes = plt.subplots(nrows, ncols, figsize = (5 * ncols, 4 * nrows))
            axes = np.array(axes).reshape(nrows, ncols)

            ref_s   = self._samples[self.ref_idx]
            tgt_s   = self._samples[i]
            gains   = np.array([r.gain   for r in band_results])
            offsets = np.array([r.offset for r in band_results])
            norm_s  = self._apply_gains_np(tgt_s, gains, offsets)

            for b in tqdm(range(self._n_bands), desc = f"Image {i+1}", dynamic_ncols = True, leave = False):
                pif_idx = np.where(self._pif_mask[b])[0]

                ref_pif  = ref_s[b][pif_idx]
                tgt_pif  = tgt_s[b][pif_idx]
                norm_pif = self._apply_gains_np(tgt_s[:, pif_idx], gains, offsets)[b]

                both_finite = np.isfinite(ref_pif) & np.isfinite(tgt_pif)
                ref_pif  = ref_pif[both_finite]
                tgt_pif  = tgt_pif[both_finite]
                norm_pif = norm_pif[both_finite]

                n_plot = min(1_000, len(ref_pif))
                if n_plot > 0:
                    sub = rng.choice(len(ref_pif), n_plot, replace=False)
                    ref_pif  = ref_pif[sub]
                    tgt_pif  = tgt_pif[sub]
                    norm_pif = norm_pif[sub]

                row_raw  = (b // ncols) * 2
                row_norm = row_raw + 1
                col      = b % ncols
                label    = self._band_labels[b]
                result   = band_results[b]

                ref_all_b  = ref_s[b][idx]
                tgt_all_b  = tgt_s[b][idx]
                norm_all_b = norm_s[b][idx]
                fin = np.isfinite(ref_all_b) & np.isfinite(tgt_all_b)
                ref_all  = ref_all_b[fin]
                tgt_all  = tgt_all_b[fin]
                norm_all = norm_all_b[fin]

                vmax_raw  = float(np.percentile(np.concatenate([ref_all, tgt_all]),  98))
                vmax_norm = float(np.percentile(np.concatenate([ref_all, norm_all]), 98))

                ax = axes[row_raw, col]

                ax.scatter(
                    ref_all,
                    tgt_all,
                    s = 1,
                    alpha = 0.15,
                    color = "steelblue",
                    rasterized = True,
                    label = "all")
                
                ax.scatter(
                    ref_pif,
                    tgt_pif,
                    s = 4,
                    alpha = 0.5,
                    color = "gold",
                    rasterized = True,
                    label = "PIFs")
                
                x = np.array([0, vmax_raw])

                ax.plot(x, x, "k--", lw = 0.8, label = "1:1")
                ax.set_title(f"Image {i+1} | Band {label} Raw \n gain = {result.gain:.3f} offset = {result.offset:+.3f}", fontsize = 9)
                ax.set_xlabel("ref", fontsize = 8)
                ax.set_ylabel("tgt_raw", fontsize = 8)
                ax.set_xlim(0, vmax_raw);  ax.set_ylim(0, vmax_raw)
                ax.legend(fontsize = 7, markerscale = 3)

                ax = axes[row_norm, col]
                
                ax.scatter(
                    ref_all,
                    norm_all,
                    s = 1,
                    alpha = 0.15,
                    color = "darkorange",
                    rasterized = True,
                    label = "all")
                
                ax.scatter(
                    ref_pif,
                    norm_pif,
                    s = 4,
                    alpha = 0.5,
                    color = "gold",
                    rasterized = True,
                    label = "PIFs")
                
                x = np.array([0, vmax_norm])
                
                ax.plot(x, x, "k--", lw = 0.8, label = "1:1")
                ax.set_title(f"Image {i+1} | Band {label} Normalised \n R2 = {result.r2:.4f}", fontsize = 9)
                ax.set_xlabel("ref", fontsize = 8)
                ax.set_ylabel("tgt_norm", fontsize = 8)
                ax.set_xlim(0, vmax_norm);  ax.set_ylim(0, vmax_norm)
                ax.legend(fontsize = 7, markerscale = 3)

            for b in range(self._n_bands, ncols * (nrows // 2)):
                axes[(b // ncols) * 2, b % ncols].set_visible(False)
                axes[(b // ncols) * 2 + 1, b % ncols].set_visible(False)

            fig.suptitle(f"PIF Normalisation: Image {i+1} -> Reference", fontsize = 13, y = 1.01)
            fig.tight_layout()

            if output is not None:
                fig.savefig(output, dpi = 150, bbox_inches = "tight")
                self._log(f"Saved: {output}")
            else:
                plt.show()
            plt.close(fig)


    def plot_mdps(self, output=None):
        self._check_fitted()

        iy, ix = self._sample_coords
        n_bands, ny, nx = self.trainers[0].shape

        pbar_b = tqdm(range(n_bands), desc="MDPS plot", unit="band", dynamic_ncols=True)
        for b in pbar_b:
            label = self._band_labels[b]
            pbar_b.set_description(f"MDPS plot | Band {label}")

            proj_1d, dist, U, valid = self._mdps_cache[b]
            pif_full  = self._pif_mask[b]
            pif_valid = pif_full[valid]

            fig, (ax_cyl, ax_map) = plt.subplots(1, 2, figsize=(14, 4))

            # --- Cylinder plot ---
            ax_cyl.scatter(proj_1d[~pif_valid], dist[~pif_valid],
                           s=1, alpha=0.15, color="steelblue", label="non-PIF", rasterized=True)
            ax_cyl.scatter(proj_1d[pif_valid], dist[pif_valid],
                           s=2, alpha=0.5, color="gold", label="PIF", rasterized=True)
            ax_cyl.axhline(U, color="red", lw=1.2, linestyle="--", label=f"threshold U = {U:.3f}")
            ax_cyl.set_xlabel("Interdate Change")
            ax_cyl.set_ylabel("Radiometric Stability")
            ax_cyl.set_title(f"Band {label} - MDPS cylinder | PIFs = {pif_valid.sum():,} | Sampled = {valid.sum():,}")
            ax_cyl.legend(fontsize=8, markerscale=3)

            # --- Spatial map ---
            spatial = np.full((ny, nx), np.nan, dtype=np.float32)
            spatial[iy[valid], ix[valid]] = 0.0
            spatial[iy[pif_full], ix[pif_full]] = 1.0
            im = ax_map.imshow(spatial, cmap="RdYlGn", vmin=0, vmax=1, interpolation="none")
            plt.colorbar(im, ax=ax_map, fraction=0.046, pad=0.04,
                         ticks=[0, 1], label="0 = Sampled | 1 = PIF")
            ax_map.set_title(f"Band {label} - PIF spatial distribution")
            ax_map.axis("off")

            fig.tight_layout()

            if output is not None:
                path = output if n_bands == 1 else str(output).replace(".", f"_band{label}.")
                fig.savefig(path, dpi=150, bbox_inches="tight")
                self._log(f"Saved: {path}")
            else:
                plt.show()
            plt.close(fig)

    def plot_residuals(self, output=None):
        self._check_fitted()

        non_ref = [i for i in range(self._n_images) if i != self.ref_idx]
        pbar_i = tqdm(non_ref, desc="Residual plots", unit="image", dynamic_ncols=True)

        for i in pbar_i:
            pbar_b = tqdm(range(self._n_bands), desc=f"Image {i+1}",
                          unit="band", leave=False, dynamic_ncols=True)
            for b in pbar_b:
                label = self._band_labels[b]
                pbar_b.set_description(f"Residuals | img {i+1} band {label}")

                pif_mask  = self._pif_mask[b]
                tgt       = self._samples[i,            b, pif_mask].astype(np.float64)
                ref       = self._samples[self.ref_idx, b, pif_mask].astype(np.float64)

                r         = self.results[i][b]
                predicted = r.gain * tgt + r.offset
                residuals = ref - predicted
                rmse      = float(np.sqrt(np.mean(residuals ** 2)))

                fig, (ax_sc, ax_res) = plt.subplots(1, 2, figsize=(12, 4))

                # --- Scatter: target PIF vs reference PIF with fit line ---
                ax_sc.hexbin(tgt, ref, gridsize=80, cmap="Blues",
                             mincnt=1, linewidths=0)
                x_line = np.array([tgt.min(), tgt.max()])
                ax_sc.plot(x_line, r.gain * x_line + r.offset,
                           color="red", lw=1.5,
                           label=f"gain={r.gain:.3f}  offset={r.offset:+.4f}")
                ax_sc.plot(x_line, x_line, color="grey", lw=0.8,
                           linestyle="--", label="1:1")
                ax_sc.set_xlabel("Target PIF reflectance")
                ax_sc.set_ylabel("Reference PIF reflectance")
                ax_sc.set_title(
                    f"Image {i+1} | Band {label}  "
                    f"R²={r.r2:.3f}  n_pifs={r.n_pifs:,}"
                )
                ax_sc.legend(fontsize=8)

                # --- Residuals vs fitted ---
                ax_res.hexbin(predicted, residuals, gridsize=80, cmap="RdBu_r",
                              mincnt=1, linewidths=0)
                ax_res.axhline(0, color="black", lw=1.0, linestyle="-")
                ax_res.set_xlabel("Fitted (gain × target + offset)")
                ax_res.set_ylabel("Residual (ref − predicted)")
                ax_res.set_title(f"Residuals vs Fitted | RMSE={rmse:.4f}")

                fig.tight_layout()

                if output is not None:
                    path = str(output).replace(".", f"_img{i+1}_band{label}.")
                    fig.savefig(path, dpi=150, bbox_inches="tight")
                    self._log(f"Saved: {path}")
                else:
                    plt.show()
                plt.close(fig)

    def quadratic_difference(self, subsample = 50_000):
        """
        Quadratic Difference index (Paolini et al. 2006). For each band, compute the PCA major-axis slope between every pair of images and report QD = sum((1 - SL)^2). QD near 0 means images are radiometrically consistent.
        """
        self._check_fitted()

        # Subsample for speed
        rng    = np.random.default_rng(self.random_state)
        n_samp = min(subsample, self._samples.shape[2])
        s_idx  = rng.choice(self._samples.shape[2], n_samp, replace=False)
        s      = self._samples[:, :, s_idx].astype(np.float32)

        norm_s = s.copy()
        for i, band_results in self.results.items():
            gains   = np.array([r.gain   for r in band_results], dtype=np.float32)
            offsets = np.array([r.offset for r in band_results], dtype=np.float32)
            norm_s[i] = np.clip(
                gains[:, np.newaxis] * s[i] + offsets[:, np.newaxis],
                0, None
            )

        N = self._n_images
        pairs       = [(i, j) for i in range(N) for j in range(i + 1, N)]
        pair_labels = [f"({i},{j})" for i, j in pairs]

        col_w  = 10
        sl_hdr = "".join(f"  {'SL'+p:>{col_w}}" for p in pair_labels)
        header = f"{'Band':<10}  {'QD':>8}{sl_hdr}"

        self._qd = {}

        def _qd_table(arrays, title):
            lines = [f"\nQuadratic Difference Index (Paolini et al. 2006) - {title}", header]
            qd_dict = {}
            pbar_b = tqdm(range(self._n_bands), desc=f"QD ({title})", unit="band", dynamic_ncols=True)
            for b in pbar_b:
                pbar_b.set_description(f"QD {title} | Band {self._band_labels[b]}")
                label = self._band_labels[b]
                sl_vals = []
                for i, j in pairs:
                    x = arrays[i, b, :]
                    y = arrays[j, b, :]
                    fin = np.isfinite(x) & np.isfinite(y)
                    x, y = x[fin], y[fin]
                    if len(x) < 4:
                        sl_vals.append(np.nan)
                        continue
                    X2d = np.column_stack([x, y])
                    X2d -= X2d.mean(axis=0)
                    pca_qd = PCA(n_components=1)
                    pca_qd.fit(X2d)
                    v  = pca_qd.components_[0]
                    sl = float(v[1] / v[0]) if abs(v[0]) > 1e-10 else np.nan
                    sl_vals.append(sl)

                qd = float(np.nansum([(1 - sl) ** 2 for sl in sl_vals]))
                qd_dict[label] = {"qd": qd, "sl": dict(zip(pair_labels, sl_vals))}

                sl_str = "".join(f"  {sl:>{col_w}.4f}" for sl in sl_vals)
                lines.append(f"{label!s:<10}  {qd:>8.4f}{sl_str}")
            return "\n".join(lines), qd_dict

        raw_str,  raw_qd  = _qd_table(s,      "Raw")
        norm_str, norm_qd = _qd_table(norm_s,  "Normalised")
        self._qd = {"raw": raw_qd, "normalised": norm_qd}

        return raw_str + "\n" + norm_str

    def plot_imgs(self, output=None, percentile=2, downsample=10):
        self._check_fitted()
        for b in range(self._n_bands):
            label = self._band_labels[b]
            fig, axes = plt.subplots(1, self._n_images, figsize=(5 * self._n_images, 5))

            all_vals = []
            for da in self.normalised:
                arr = da.isel(band=b).values[::downsample, ::downsample]
                arr = arr[np.isfinite(arr)]
                all_vals.append(arr)
            all_vals = np.concatenate(all_vals)
            vmin = float(np.percentile(all_vals, percentile))
            vmax = float(np.percentile(all_vals, 100 - percentile))

            for i, da in enumerate(self.normalised):
                ax = axes[i]
                arr = da.isel(band=b).values[::downsample, ::downsample]
                ref_marker = " (ref)" if i == self.ref_idx else ""
                im = ax.imshow(arr, vmin=vmin, vmax=vmax, cmap="viridis", interpolation="none")
                ax.set_title(f"Image {i + 1}{ref_marker}", fontsize=9)
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

            fig.suptitle(f"Band {label} - Normalised ([{vmin:.4f}, {vmax:.4f}])", fontsize=11)
            fig.tight_layout()

            if output is not None:
                path = output if self._n_bands == 1 else str(output).replace(".", f"_band{label}.")
                fig.savefig(path, dpi=150, bbox_inches="tight")
                self._log(f"Saved -> {path}")
            else:
                plt.show()
            plt.close(fig)

NOTE: Use Pandas for summaries, etc.

In [7]:
def load_item_for_normalisation(
    item,
    input_asset = None,
    bands=NORM_BANDS,
    passthrough=PASSTHROUGH,
    chunks=CHUNKS,
    verbose=VERBOSE):

    if INPUT_ASSET is None:
        input_asset = INPUT_ASSET[item.collection_id]

    bands_idx, band_names = bands[item.collection_id]

    target = rxr.open_rasterio(item.assets[input_asset].href, chunks=chunks)
    target = target.sel(band=bands_idx).assign_coords(band=band_names)
    target = target.where(target != target.rio.nodata)

    # Masks (always applied to target/trainer)
    masks_loaded = []

    if "saturation-mask" in item.assets and Path(item.assets["saturation-mask"].href).exists():
        sat = rxr.open_rasterio(item.assets["saturation-mask"].href, chunks=chunks)
        sat = sat.sel(band=bands_idx).assign_coords(band=band_names)
        target = target.where(~sat.any(dim="band"))
        masks_loaded.append("saturation")

    if "edge-mask" in item.assets and Path(item.assets["edge-mask"].href).exists():
        edge = rxr.open_rasterio(item.assets["edge-mask"].href, chunks=chunks)
        edge = edge.sel(band=bands_idx).assign_coords(band=band_names)
        target = target.where(~edge.any(dim="band"))
        masks_loaded.append("edge")

    if "invalid-mask" in item.assets and Path(item.assets["invalid-mask"].href).exists():
        invalid = rxr.open_rasterio(item.assets["invalid-mask"].href, chunks=chunks).squeeze()
        target = target.where(~invalid)
        masks_loaded.append("invalid")

    # Training on non-veg land only
    trainer = target
    if "water-mask" in item.assets and Path(item.assets["water-mask"].href).exists():
        water = rxr.open_rasterio(item.assets["water-mask"].href, chunks=chunks).squeeze()
        trainer = trainer.where(~water)
        masks_loaded.append("water")

    if "veg-mask" in item.assets and Path(item.assets["veg-mask"].href).exists():
        veg = rxr.open_rasterio(item.assets["veg-mask"].href, chunks=chunks).squeeze()
        trainer = trainer.where(~veg)
        masks_loaded.append("veg")

    # Passthrough bands
    pt = None
    if item.collection_id in passthrough:
        pt_idx, pt_names = passthrough[item.collection_id]
        if pt_idx:
            pt = rxr.open_rasterio(item.assets[input_asset].href, chunks=chunks)
            pt = pt.sel(band=pt_idx).assign_coords(band=pt_names)
            pt = pt.where(pt != pt.rio.nodata)
            if "saturation-mask" in item.assets and Path(item.assets["saturation-mask"].href).exists():
                sat_pt = rxr.open_rasterio(item.assets["saturation-mask"].href, chunks=chunks)
                sat_pt = sat_pt.sel(band=pt_idx).assign_coords(band=pt_names)
                pt = pt.where(~sat_pt.any(dim="band"))
            if "edge-mask" in item.assets and Path(item.assets["edge-mask"].href).exists():
                pt = pt.where(~edge.any(dim="band"))
            if "invalid-mask" in item.assets and Path(item.assets["invalid-mask"].href).exists():
                pt = pt.where(~invalid)

    if verbose:
        tqdm.write(f"{item.id} | Bands: {band_names} | Masks: {masks_loaded} | Passthrough: {pt_names if pt is not None else []}")

    return trainer, target, pt



In [8]:
wv2_targets, wv2_trainers, wv2_passthrough = [], [], []
sd8_targets, sd8_trainers, sd8_passthrough = [], [], []

pbar_load = tqdm(all_items, desc="Loading", unit="img", dynamic_ncols=True)
for item in pbar_load:
    pbar_load.set_description(f"Loading {item.id}")
    trainer, target, passthrough = load_item_for_normalisation(item, input_asset = INPUT_ASSET[item.collection_id], verbose = True)

    if item.collection_id == "wv2-imagery":
        wv2_targets.append(target)
        wv2_trainers.append(trainer)
        wv2_passthrough.append(passthrough)
    
    elif item.collection_id == "sd8-imagery":
        sd8_targets.append(target)
        sd8_trainers.append(trainer)
        sd8_passthrough.append(passthrough)

wv2_ref_idx = next(i for i, item in enumerate(sd8_items) if item.datetime.year == 2022)
sd8_ref_idx = next(i for i, item in enumerate(sd8_items) if item.datetime.year == 2022)

norm_subsets = [
    ("wv2-imagery", wv2_trainers, wv2_targets, wv2_ref_idx, wv2_items, wv2_passthrough),
    ("sd8-imagery", sd8_trainers, sd8_targets, sd8_ref_idx, sd8_items, sd8_passthrough)
]

normalisers = {}
pbar_norm = tqdm(norm_subsets, desc = "Normalisng", unit = "subset", dynamic_ncols = True)
for collection_id, trainers, targets, ref_idx, items, passthrough in pbar_norm:
    tqdm.write(f"\n=================================== {collection_id.upper()} ====================================")
    pbar_norm.set_description(f"Normalising {collection_id}")

    # --- Overwrite check ---
    out_paths = [OUT_DIR / f"{item.id}-{OUT_NAME}.{OUT_EXT}" for item in items]

    if all(p.exists() for p in out_paths) and not OVERWRITE:
        tqdm.write(f"Skipping (all exist): {collection_id.upper()}")
        continue

    normaliser = PIFNormaliser(
        trainers = trainers,
        targets = targets,
        fit_sample = FIT_SAMPLE,
        cylinder_max_growth = CYLINDER_MAX_GROWTH,
        cylinder_grow_rate = CYLINDER_GROW_RATE,
        pif_fraction = PIF_FRACTION,
        ref_idx = ref_idx
    )

    normaliser.normalise()

    if collection_id in PASSTHROUGH and PASSTHROUGH[collection_id][0]:
        pt_names = list(PASSTHROUGH[collection_id][1])
        norm_names = list(NORM_BANDS[collection_id][1])
        all_names = set(norm_names + pt_names)
        full_order = [b for b in ALL_BAND_NAMES[collection_id] if b in all_names]
    
        normaliser.normalised = [
            xr.concat([norm.astype(np.float32), pt.astype(np.float32)], dim="band")
            .sel(band=full_order)
            for norm, pt in zip(normaliser.normalised, passthrough)
        ]

    normalisers[collection_id] = normaliser

results = {collection_id: n.normalised for collection_id, n in normalisers.items()}

Loading:   0%|          | 0/8 [00:00<?, ?img/s]

wv2-20110610 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'edge', 'invalid', 'water', 'veg'] | Passthrough: []
wv2-20120620 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'edge', 'invalid', 'water', 'veg'] | Passthrough: []
wv2-20140714 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'edge', 'invalid', 'water', 'veg'] | Passthrough: []
wv2-20150701 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'edge', 'invalid', 'water', 'veg'] | Passthrough: []
sd8-20220713 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'invalid', 'water', 'veg'] | Passthrough: []
sd8-20230706 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'invalid', 'water', 'veg'] | Passthrough: []
sd8-20240722 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'invalid', 'water', 'veg'] | Passthrough: []
sd8-20250610 | Bands: ['B', 'G', 'Y', 'R'] | Masks: ['saturation', 'invalid', 'water', 'veg'] | Passthrough: []


Normalisng:   0%|          | 0/2 [00:00<?, ?subset/s]


=================================== WV2-IMAGERY ====================================


  0%|          | 0/4 [00:00<?, ?it/s]

...

Growing cylinder: 0step [00:00, ?step/s]

Normalising images:   0%|          | 0/4 [00:00<?, ?img/s]

Normalising bands:   0%|          | 0/4 [00:00<?, ?band/s]

Normalising bands:   0%|          | 0/4 [00:00<?, ?band/s]

Normalising bands:   0%|          | 0/4 [00:00<?, ?band/s]

In [9]:
for collection_id, n in normalisers.items():
    tqdm.write(f"\n=================================== {collection_id.upper()} ====================================")
    tqdm.write(n.summary())
    tqdm.write(n.quadratic_difference())


=================================== WV2-IMAGERY ====================================

PIF Normalisation Summary | n = 4 | ref_idx = 0 | candidates = 34474038 | sampled = 34474038

B          | 315,267 PIFs (0.9 %)
G          | 315,267 PIFs (0.9 %)
Y          | 315,267 PIFs (0.9 %)
R          | 315,267 PIFs (0.9 %)

Image 2:
Band      B | gain = 0.9385 | offset = +0.0016 | r2 = 0.9407 | n PIFs = 258,655
Band      G | gain = 1.1611 | offset = -0.0065 | r2 = 0.9590 | n PIFs = 305,366
Band      Y | gain = 1.0744 | offset = -0.0003 | r2 = 0.9571 | n PIFs = 307,182
Band      R | gain = 2.1937 | offset = -0.0120 | r2 = 0.9075 | n PIFs = 238,397
Image 3:
Band      B | gain = 1.1924 | offset = -0.0431 | r2 = 0.9681 | n PIFs = 270,371
Band      G | gain = 1.5861 | offset = -0.0410 | r2 = 0.9496 | n PIFs = 280,555
Band      Y | gain = 1.2919 | offset = -0.0060 | r2 = 0.9528 | n PIFs = 314,648
Band      R | gain = 1.8789 | offset = -0.0094 | r2 = 0.7062 | n PIFs = 216,355
Image 4:
Band      B | g

QD (Raw):   0%|          | 0/4 [00:00<?, ?band/s]

QD (Normalised):   0%|          | 0/4 [00:00<?, ?band/s]


Quadratic Difference Index (Paolini et al. 2006) - Raw
Band              QD     SL(0,1)     SL(0,2)     SL(0,3)     SL(1,2)     SL(1,3)     SL(2,3)
B             0.2768      0.9406      0.7624      0.6843      0.8128      0.7266      0.9141
G             0.3959      0.8155      0.6026      0.7142      0.7445      0.8813      1.2070
Y             0.2797      0.8966      0.7548      0.6577      0.8504      0.7598      0.8924
R             6.0134      0.0924      0.0466      0.1239      0.6021      1.6099      2.7271

Quadratic Difference Index (Paolini et al. 2006) - Normalised
Band              QD     SL(0,1)     SL(0,2)     SL(0,3)     SL(1,2)     SL(1,3)     SL(2,3)
B             0.0393      0.8771      0.9357      0.9800      1.0705      1.1116      1.0470
G             0.0100      0.9597      1.0123      1.0263      1.0493      1.0700      1.0134
Y             0.0185      0.9687      1.0269      1.0662      1.0431      1.0968      1.0351
R             6.9161      0.2977      0.0952

QD (Raw):   0%|          | 0/4 [00:00<?, ?band/s]

QD (Normalised):   0%|          | 0/4 [00:00<?, ?band/s]


Quadratic Difference Index (Paolini et al. 2006) - Raw
Band              QD     SL(0,1)     SL(0,2)     SL(0,3)     SL(1,2)     SL(1,3)     SL(2,3)
B             0.0961      1.2205      1.1486      1.1242      0.9418      0.9216      0.9805
G             0.0302      1.1141      1.0561      1.0995      0.9499      0.9845      1.0367
Y             0.0688      0.9728      0.8323      0.9157      0.8591      0.9436      1.0988
R             0.0704      1.1245      0.9350      0.9523      0.8378      0.8526      1.0195

Quadratic Difference Index (Paolini et al. 2006) - Normalised
Band              QD     SL(0,1)     SL(0,2)     SL(0,3)     SL(1,2)     SL(1,3)     SL(2,3)
B             0.0399      0.9018      0.8698      0.8907      0.9711      0.9922      1.0220
G             0.0079      0.9713      0.9493      0.9424      0.9790      0.9726      0.9942
Y             0.4639      0.6588      0.9414      0.6650      1.3944      1.0069      0.7238
R             0.0274      0.9324      0.9005

In [10]:
def write_norm(out_path, da, profile_options = PROFILE_FLOAT32):
    n_bands, h, w = da.shape
    crs = da.rio.crs
    transform = da.rio.transform()

    profile = dict(
        height = h,
        width = w,
        count = n_bands,
        crs = crs,
        transform = transform,
        **profile_options
    )

    with rasterio.open(out_path, "w", **profile) as dst:
        pbar_band = tqdm(total=n_bands, desc = "Writing band", unit = "band", leave = True)
        with pbar_band:
            for i in range(n_bands):
                pbar_band.set_description(f"Writing band {i + 1}")
                dst.write(da.isel(band=i).values, i + 1)
                pbar_band.update(1)

    with rasterio.open(out_path, "r+") as dst:
        pbar_ovr = tqdm(total = 1, desc = "Building overviews", leave = True)
        with pbar_ovr:
            dst.build_overviews([2, 4, 8, 16], Resampling.nearest)
            dst.update_tags(ns="rio_overview", resampling = "nearest")
            pbar_ovr.update(1)

In [11]:
for collection_id, n in normalisers.items():
    tqdm.write(f"\n=================================== {collection_id.upper()} ====================================")
    n.plot_diag()

In [12]:
for collection_id, n in normalisers.items():
    tqdm.write(f"\n=================================== {collection_id.upper()} ====================================")
    n.plot_residuals()

NOTE: `mpds_plot()`

Each dot is a sampled pixel. The x-axis is its overall brightness relative to the scene (in standardised units); the y-axis is how much its reflectance pattern across dates deviates from the dominant inter-date trend.

Pixels below the dashed red line (threshold U = MDPS cylinder radius) are those with small variation across dates and are selected as PIFs; these are persistently dark or persistently bright pixels whose reflectance doesn't change in unexpected ways across acquisitions. The dual-lobe fan shape reflects two stable populations in the visible range: consistently dark surfaces (left) and consistently bright surfaces (right), with variable middle-SR pixels in between. NIR have single-lobed distributions with much higher radial spread, centred around x = 0: no stable populations as such. Higher residual variance means that the cylinder radius ends up larger to capture the same pif_fraction of pixels for PIFs. Hence the most "stable" 5% sit further from the axis than their visible counterparts. Risk of fitting noise comes from the wider U.

Right panels: Spatial distribution of selected PIFs (green) vs other sampled pixels (red). NaN pixels (water, cloud, edge-contaminated) are not sampled and appear white.

NEEDS REWRITE



In [13]:
# for sensor, n in normalisers.items():
#     tqdm.write(f"\n=================================== {sensor.upper()} ====================================")
#     n.plot_mdps()

In [14]:
for collection_id, _, _, _, items, _ in norm_subsets:
    normaliser = normalisers.get(collection_id)

    pbar_write = tqdm(enumerate(items, start = 1), unit = "scene", desc = f"Writing {collection_id.upper()}", dynamic_ncols = True, total = len(items))

    for i, item in pbar_write:
        pbar_write.set_description(f"[{i}/{len(items)}] Writing {item.id}")
        out_path = OUT_DIR / f"{item.id}-{OUT_NAME}.{OUT_EXT}"

        asset = pystac.Asset(
            href = str(out_path.resolve()),
            media_type = pystac.MediaType.GEOTIFF,
            roles = ["data"],
            title = "Normalised Scene"
        )

        item.add_asset(f"{OUT_NAME}", asset)

        if out_path.exists() and not OVERWRITE:
            tqdm.write(f"Skipping (exists): {out_path.name}")
            continue

        da = normaliser.normalised[i - 1]
        write_norm(out_path, da, profile_options = PROFILE_FLOAT32)
        tqdm.write(f"Saved: {out_path.name}")

CATALOG.normalize_hrefs(str(STAC_DIR))
CATALOG.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
tqdm.write("Complete")

Writing WV2-IMAGERY:   0%|          | 0/4 [00:00<?, ?scene/s]

Skipping (exists): wv2-20110610-norm.tif
Skipping (exists): wv2-20120620-norm.tif
Skipping (exists): wv2-20140714-norm.tif
Skipping (exists): wv2-20150701-norm.tif


Writing SD8-IMAGERY:   0%|          | 0/4 [00:00<?, ?scene/s]

Skipping (exists): sd8-20220713-norm.tif
Skipping (exists): sd8-20230706-norm.tif
Skipping (exists): sd8-20240722-norm.tif
Skipping (exists): sd8-20250610-norm.tif
Complete


# Plot

In [8]:
records = []

INPUT_ASSETS =          {
                            "wv2-imagery" : "sat-corr",
                            "sd8-imagery" : "harm",
                        }

DROP_BANDS = {
                "wv2-imagery" : ["CB", "REdge", "NIR1", "NIR2"],
                "sd8-imagery" : ["CB", "REdge", "NIR1"]
            }

for item in all_items:
    img_col = item.collection_id
    band_idx, band_names = NORM_BANDS[img_col]
    keep_names = [n for n in band_names if n not in DROP_BANDS[img_col]]

    raw_da  = rxr.open_rasterio(item.assets[INPUT_ASSETS[img_col]].href, chunks=CHUNKS, masked=True)
    norm_da = rxr.open_rasterio(item.assets["norm"].href,                chunks=CHUNKS, masked=True)

    raw_arr  = raw_da.values[[i-1 for i in band_idx]].astype(np.float32)
    norm_arr = norm_da.values.astype(np.float32)

    print(raw_arr.shape)
    print(norm_arr.shape)




(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)
(4, 7667, 4600)


In [9]:
import matplotlib.colors as mcolors
import matplotlib.cm as mcm

records = []
waves_labels = ["B", "G", "Y", "R"]

for item in all_items:
    img_col = item.collection_id
    band_idx, band_names = NORM_BANDS[img_col]

    raw_da  = rxr.open_rasterio(item.assets[INPUT_ASSETS[img_col]].href, chunks=CHUNKS, masked=True)
    norm_da = rxr.open_rasterio(item.assets["norm"].href,                chunks=CHUNKS, masked=True)

    raw_arr  = raw_da.values[[i-1 for i in band_idx]].astype(np.float32)
    norm_arr = norm_da.values.astype(np.float32)

    valid = np.isfinite(raw_arr).all(axis=0) & np.isfinite(norm_arr).all(axis=0)
    if valid.sum() < 10:
        continue

    year   = item.id.split("-")[1][:4]
    sensor = "WV2" if img_col == "wv2-imagery" else "SD8"

    for band, v in zip(waves_labels, np.median(raw_arr[:, valid], axis=1)):
        records.append({"band": band, "reflectance": v,
                        "label": f"{sensor} {year} (Before)",  "sensor": sensor, "year": year})
    for band, v in zip(waves_labels, np.median(norm_arr[:, valid], axis=1)):
        records.append({"band": band, "reflectance": v,
                        "label": f"{sensor} {year} (After)", "sensor": sensor, "year": year})


In [16]:
from plotnine import (
    ggplot, aes, geom_line,
    scale_color_manual, scale_linetype_manual,
    labs, theme_bw, theme, element_text, scale_x_continuous, facet_wrap, guides, ggsave
)

df = pd.DataFrame(records)
df["band"] = pd.Categorical(df["band"], categories=waves_labels, ordered=True)

wv2_years = sorted(df.query("sensor=='WV2'")["year"].unique())
sd8_years = sorted(df.query("sensor=='SD8'")["year"].unique())

year_color = {
    **dict(zip(wv2_years, [mcolors.to_hex(mcm.Blues(v)) for v in np.linspace(0.45, 0.85, len(wv2_years))])),
    **dict(zip(sd8_years, [mcolors.to_hex(mcm.Reds(v))  for v in np.linspace(0.35, 0.6,  len(sd8_years))])),
}
year_color["2022"] = "#8B0000"

color_map = {r.label: year_color[r.year] for r in df.drop_duplicates(["label","year"]).itertuples()}
linetype_map = {lbl: ("dashed" if "(Before)" in lbl else "solid") for lbl in df["label"].unique()}

df["type"] = df["label"].apply(lambda x: "Before" if "(Before)" in x else "After")
df["type"] = pd.Categorical(df["type"], categories=["Before", "After"], ordered=True)

p = (
    ggplot(df, aes("band", "reflectance", color="label", group="label"))
    + geom_line(size=0.6)
    + scale_color_manual(values=color_map)
    + facet_wrap("type", ncol=2)
    + labs(x="Band", y="Reflectance")
    + guides(color=False)
    + theme_bw()
    + theme(figure_size=(7.33/2.54, 6/2.54),
            text=element_text(size=7),
            axis_text=element_text(size=6),
            axis_title=element_text(size=7)
))

ggsave(p, "../out/figs/raw_vs_norm.png", dpi=300)

c:\Users\samla\.conda\envs\geospat\Lib\site-packages\plotnine\ggplot.py:623: PlotnineWarning: Saving 2.8858267716535435 x 2.3622047244094486 in image.
c:\Users\samla\.conda\envs\geospat\Lib\site-packages\plotnine\ggplot.py:624: PlotnineWarning: Filename: ../out/figs/raw_vs_norm.png


In [12]:
print(df["label"].unique())
print(df["type"].value_counts())

['WV2 2011 (Before)' 'WV2 2011 (After)' 'WV2 2012 (Before)'
 'WV2 2012 (After)' 'WV2 2014 (Before)' 'WV2 2014 (After)'
 'SD8 2022 (Before)' 'SD8 2022 (After)' 'SD8 2023 (Before)'
 'SD8 2023 (After)' 'SD8 2024 (Before)' 'SD8 2024 (After)'
 'SD8 2025 (Before)' 'SD8 2025 (After)']
type
norm    56
raw      0
Name: count, dtype: int64
